# Aula 14 — Limpeza de dados e valores ausentes

**Módulo 4 — Pandas e Análise de Dados**

## Objetivos da aula

- Identificar e tratar valores ausentes em um `DataFrame`.
- Padronizar textos e corrigir inconsistências em colunas categóricas.
- Detectar e remover linhas duplicadas.

---

## 1. Dados reais raramente chegam limpos

Até aqui trabalhamos com uma base "perfeita". Na prática, os dados que chegam de sensores e sistemas reais costumam ter leituras ausentes (sensor que falhou), textos digitados de formas diferentes (`"Normal"`, `"normal "`, `"NORMAL"`) e até registros duplicados. Vamos recriar nossa base e, propositalmente, "sujá-la", para praticar as técnicas de limpeza.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500
tipos_equipamento = ["Motor", "Bomba", "Compressor", "Ventilador"]

dados = pd.DataFrame({
    "equipamento_id": [f"EQ-{i:04d}" for i in range(1, n + 1)],
    "tipo_equipamento": np.random.choice(tipos_equipamento, size=n),
    "temperatura": np.round(np.random.normal(70, 12, size=n), 1),
    "pressao": np.round(np.random.normal(5.5, 1.3, size=n), 2),
    "vibracao": np.round(np.random.normal(2.4, 1.1, size=n), 2),
    "horas_operacao": np.random.randint(0, 10000, size=n),
})


def definir_status(linha):
    critico = (linha["temperatura"] >= 90) or (linha["vibracao"] >= 4.5) or (linha["pressao"] >= 8) or (linha["pressao"] <= 2)
    alerta = (linha["temperatura"] >= 80) or (linha["vibracao"] >= 3.5) or (linha["pressao"] >= 7) or (linha["pressao"] <= 3)
    if critico:
        return "critico"
    elif alerta:
        return "alerta"
    else:
        return "normal"


dados["status"] = dados.apply(definir_status, axis=1)
dados.to_csv("sensores_industriais.csv", index=False)

df = pd.read_csv("sensores_industriais.csv")
df.head()


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


In [2]:
import numpy as np

df_bruto = df.copy()   # trabalhamos sempre em uma cópia, preservando os dados originais

rng = np.random.default_rng(7)

# 1) introduzindo valores ausentes em colunas numéricas
indices_nan_temperatura = rng.choice(df_bruto.index, size=15, replace=False)
df_bruto.loc[indices_nan_temperatura, "temperatura"] = np.nan

indices_nan_pressao = rng.choice(df_bruto.index, size=10, replace=False)
df_bruto.loc[indices_nan_pressao, "pressao"] = np.nan

# 2) "sujando" a coluna categórica com variações de escrita
variacoes = {"normal": ["normal", "Normal", "NORMAL ", " normal"],
             "alerta": ["alerta", "Alerta", "ALERTA "],
             "critico": ["critico", "Critico", "CRITICO "]}
df_bruto["status"] = df_bruto["status"].apply(lambda s: rng.choice(variacoes[s]))

# 3) duplicando algumas linhas, como acontece quando um mesmo dado é importado duas vezes
df_bruto = pd.concat([df_bruto, df_bruto.sample(8, random_state=7)], ignore_index=True)

print(f"Base 'suja' com {len(df_bruto)} linhas (o dobro de duplicatas propositais + valores ausentes).")
df_bruto.head()


Base 'suja' com 508 linhas (o dobro de duplicatas propositais + valores ausentes).


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,Normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,NaN,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,Alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


## 2. Identificando valores ausentes

`isna()` (ou `isnull()`, equivalente) retorna `True`/`False` para cada célula. Combinado com `.sum()`, conta quantos ausentes existem em cada coluna.

In [3]:
print("Valores ausentes por coluna:")
print(df_bruto.isna().sum())

print("\nTotal de linhas com pelo menos um valor ausente:", df_bruto.isna().any(axis=1).sum())


Valores ausentes por coluna:
equipamento_id       0
tipo_equipamento     0
temperatura         16
pressao             10
vibracao             0
horas_operacao       0
status               0
dtype: int64

Total de linhas com pelo menos um valor ausente: 26


## 3. Tratando valores ausentes

Duas estratégias principais:

- **`dropna()`**: remove as linhas (ou colunas) com valores ausentes. Simples, mas descarta dados que podem ser úteis.
- **`fillna()`**: preenche os ausentes com algum valor (a média da coluna, por exemplo), preservando a linha.

In [4]:
# remove qualquer linha que tenha ao menos um valor ausente
df_sem_ausentes = df_bruto.dropna()
print(f"Linhas antes: {len(df_bruto)} | depois do dropna(): {len(df_sem_ausentes)}")


Linhas antes: 508 | depois do dropna(): 482


In [5]:
# preenchendo os ausentes com a média da própria coluna, em vez de descartar a linha
df_preenchido = df_bruto.copy()
df_preenchido["temperatura"] = df_preenchido["temperatura"].fillna(df_preenchido["temperatura"].mean())
df_preenchido["pressao"] = df_preenchido["pressao"].fillna(df_preenchido["pressao"].mean())

print("Valores ausentes após fillna():")
print(df_preenchido[["temperatura", "pressao"]].isna().sum())


Valores ausentes após fillna():
temperatura    0
pressao        0
dtype: int64


**Qual usar?** `dropna()` é adequado quando poucos registros têm problema e a base é grande o suficiente para não fazer falta. `fillna()` é preferível quando não podemos "perder" a linha — por exemplo, se as outras colunas daquele registro são valiosas. A melhor escolha depende do contexto e será revisitada no Módulo 6.

## 4. Padronizando texto

A coluna `status` está com variações de maiúsculas/minúsculas e espaços em branco. Os métodos de string do Pandas (acessados via `.str`) resolvem isso de forma vetorizada, sem precisar de laços.

In [ ]:
print("Valores únicos antes da limpeza:")
print(sorted(df_bruto["status"].unique()))

df_preenchido["status"] = df_preenchido["status"].str.strip().str.lower()

print("\nValores únicos depois de strip() + lower():")
print(sorted(df_preenchido["status"].unique()))


Valores únicos antes da limpeza:
[' normal', 'ALERTA ', 'Alerta', 'CRITICO ', 'Critico', 'NORMAL ', 'Normal', 'alerta', 'critico', 'normal']

Valores únicos depois de strip() + lower():
['alerta', 'crítico', 'normal']


### `replace()`: corrigindo valores específicos

Quando a inconsistência não é apenas de formatação, mas de **nomenclatura** (por exemplo, `"critico"` vs. o nome que queremos padronizar, `"crítico"`), usamos `replace()` com um dicionário de correspondências.

In [7]:
df_preenchido["status"] = df_preenchido["status"].replace({
    "normal": "normal",
    "alerta": "alerta",
    "critico": "crítico",
})

print(df_preenchido["status"].value_counts())


status
normal     282
alerta     164
crítico     62
Name: count, dtype: int64


## 5. Identificando e removendo duplicatas

`duplicated()` sinaliza linhas repetidas; `drop_duplicates()` as remove, mantendo apenas a primeira ocorrência por padrão.

In [8]:
print("Linhas duplicadas encontradas:", df_preenchido.duplicated().sum())

df_limpo = df_preenchido.drop_duplicates()
print(f"Linhas antes: {len(df_preenchido)} | depois de drop_duplicates(): {len(df_limpo)}")


Linhas duplicadas encontradas: 8
Linhas antes: 508 | depois de drop_duplicates(): 500


## 6. Conferindo o resultado final

Depois da limpeza, vale sempre conferir que os problemas foram resolvidos.

In [9]:
print("Valores ausentes restantes:")
print(df_limpo.isna().sum().sum(), "no total")

print("\nDuplicatas restantes:", df_limpo.duplicated().sum())

print("\nCategorias de status, já padronizadas:")
print(df_limpo["status"].value_counts())


Valores ausentes restantes:
0 no total

Duplicatas restantes: 0

Categorias de status, já padronizadas:
status
normal     280
alerta     161
crítico     59
Name: count, dtype: int64


## 7. Resumo da aula

- `isna().sum()` mostra a quantidade de valores ausentes por coluna.
- `dropna()` remove registros incompletos; `fillna()` preenche os ausentes preservando a linha.
- `.str.strip()`, `.str.lower()` e `.replace()` padronizam colunas categóricas com inconsistências de texto.
- `duplicated()` e `drop_duplicates()` identificam e removem linhas repetidas.
- Limpeza de dados é sempre um processo de **decisão**: cada tratamento tem prós e contras que dependem do contexto.

### Exercício sugerido

A partir de `df_bruto`, escreva o processo completo de limpeza em sequência (preencher ausentes, padronizar texto, remover duplicatas) e confirme, ao final, que `isna().sum().sum()` é `0` e `duplicated().sum()` também é `0`.
